# Mapa Interativo das Estações de Qualidade do Ar do Rio de Janeiro

Este notebook cria um mapa interativo com as estações de qualidade do ar, usando latitude e longitude

**O que será feito:**
- Carregar a base com coordenadas geográficas.
- Plotar os pontos em um mapa com camadas de rua e satélite.

## 1. Dependências

Se estiver rodando este notebook em um ambiente novo, instale as bibliotecas abaixo uma única vez.


In [1]:
from pathlib import Path
import pandas as pd
import folium
from folium import plugins
from folium.plugins import MarkerCluster
from IPython.display import HTML, display

## 2. Carregamento da base de hospitais

A leitura prioriza o arquivo local do projeto


In [2]:
url_air_quality_data = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/refs/heads/Refactoring-And-Documentation/Data/IntermediaryData/MonitorAr/InitialMergedData/merged_air_quality_data.csv"

df_air_quality = pd.read_csv(url_air_quality_data)
display(df_air_quality)

,data,Nome,chuva,temp,ur,so2,no2,co,no,nox,o3,pm10,pm2_5,lat,lon
0,2012-01-01 00:30:00,ESTAÇÃO BANGU,0.2,24.67,95.24,NaN,15.18,0.42,2.18,17.36,28.06,81.0,NaN,-22.887910,-43.471074
1,2012-01-01 00:30:00,ESTAÇÃO PEDRA DE GUARATIBA,0.8,24.07,99.56,NaN,NaN,NaN,NaN,NaN,24.52,74.0,NaN,-23.004379,-43.629010
2,2012-01-01 00:30:00,ESTAÇÃO CENTRO,0.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-22.908344,-43.178152
3,2012-01-01 00:30:00,ESTAÇÃO SÃO CRISTÓVÃO,0.2,27.14,99.82,8.41,NaN,0.17,NaN,NaN,10.95,49.0,NaN,-22.897771,-43.221745
4,2012-01-01 00:30:00,ESTAÇÃO TIJUCA,0.0,21.47,98.40,1.51,NaN,0.06,NaN,NaN,24.55,36.0,NaN,-22.924915,-43.232657
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
532687,2018-12-31 23:30:00,ESTAÇÃO COPACABANA,0.0,NaN,NaN,NaN,NaN,0.01,NaN,NaN,11.87,53.0,NaN,-22.965004,-43.180482
532688,2018-12-31 23:30:00,ESTAÇÃO CENTRO,0.0,27.23,83.68,NaN,NaN,0.66,NaN,NaN,24.68,29.0,NaN,-22.908344,-43.178152
532689,2018-12-31 23:30:00,ESTAÇÃO BANGU,0.0,26.45,99.13,1.74,40.71,1.28,3.28,44.00,3.82,80.0,NaN,-22.887910,-43.471074
532690,2018-12-31 23:30:00,ESTAÇÃO SÃO CRISTÓVÃO,0.0,27.23,NaN,NaN,NaN,1.02,NaN,NaN,4.35,49.0,NaN,-22.897771,-43.221745


## 3. Filtragem e preparação dos pontos

Nesta etapa, filtramos apenas os hospitais de interesse e mantemos um ponto único por `CNES`.
Também validamos coordenadas para evitar marcadores inválidos no mapa.


In [3]:
df_unique_estacoes = (
    df_air_quality[["Nome", "lat", "lon"]]
    .dropna(subset=["lat", "lon"])
    .drop_duplicates(subset=["Nome", "lat", "lon"])
    .sort_values("Nome")
    .reset_index(drop=True)
)

## 4. Mapa interativo dos hospitais

O mapa abaixo permite inspeção visual com zoom, troca de camada e clique nos marcadores.
Cada popup contém links para validação rápida da localização.


In [4]:
def gerar_links_validacao(lat, lon):
    return {
        "google_maps": f"https://www.google.com/maps?q={lat},{lon}",
        "street_view": f"https://www.google.com/maps/@?api=1&map_action=pano&viewpoint={lat},{lon}",
        "openstreetmap": f"https://www.openstreetmap.org/?mlat={lat}&mlon={lon}#map=18/{lat}/{lon}",
    }


def criar_popup_hospital(nome, lat, lon):
    links = gerar_links_validacao(lat, lon)
    html = (
        "<div style='font-size:13px;'>"
        f"<b>Nome:</b> {nome}<br>"
        f"<b>Latitude:</b> {lat:.6f}<br>"
        f"<b>Longitude:</b> {lon:.6f}<br><br>"
        f"<a href='{links['google_maps']}' target='_blank'>Abrir no Google Maps</a><br>"
        f"<a href='{links['street_view']}' target='_blank'>Abrir no Street View</a><br>"
        f"<a href='{links['openstreetmap']}' target='_blank'>Abrir no OpenStreetMap</a>"
        "</div>"
    )
    return html


def criar_mapa_hospitais(df_pontos, nome_destaque=None, zoom_inicial=11):
    if df_pontos.empty:
        raise ValueError("O DataFrame está vazio. Não há hospitais para plotar.")

    centro = [df_pontos["lat"].mean(), df_pontos["lon"].mean()]
    mapa = folium.Map(location=centro, zoom_start=zoom_inicial, control_scale=True, tiles=None)

    folium.TileLayer("OpenStreetMap", name="Mapa de ruas", show=True).add_to(mapa)
    folium.TileLayer(
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
        attr="Tiles (c) Esri",
        name="Satélite (Esri)",
        show=False,
    ).add_to(mapa)

    cluster = MarkerCluster(name="Estações").add_to(mapa)

    nome_destaque = str(nome_destaque) if nome_destaque is not None else None

    for row in df_pontos.itertuples(index=False):
        nome = str(row.Nome)
        lat = float(row.lat)
        lon = float(row.lon)
        destaque = nome == nome_destaque

        marker = folium.Marker(
            location=[lat, lon],
            tooltip=f"Nome {nome}",
            popup=folium.Popup(criar_popup_hospital(nome, lat, lon), max_width=360),
            icon=folium.Icon(color="red" if destaque else "blue", icon="plus-sign"),
        )
        marker.add_to(cluster)

        if destaque:
            folium.Circle(
                location=[lat, lon],
                radius=180,
                color="red",
                fill=True,
                fill_opacity=0.12,
                weight=2,
            ).add_to(mapa)
            mapa.location = [lat, lon]
            mapa.zoom_start = 16

    plugins.Fullscreen(position="topleft").add_to(mapa)
    plugins.MeasureControl(position="topleft", primary_length_unit="meters").add_to(mapa)
    plugins.MousePosition(position="bottomright").add_to(mapa)
    folium.LayerControl(collapsed=False).add_to(mapa)

    return mapa

In [5]:
mapa_hospitais = criar_mapa_hospitais(df_unique_estacoes)
mapa_hospitais